# Industrial-VLM: QLoRA Fine-Tuning Pipeline
**Model**: LLaVA-1.5-7B | **Method**: QLoRA (4-bit NF4) | **Dataset**: MVTec AD (15 categories)

This notebook runs the complete pipeline:
1. Data preprocessing (RGB conversion, 336×336 resize, stratified split)
2. Zero-shot baseline evaluation
3. QLoRA fine-tuning (r=32, α=64, Q-K-V-O projections)
4. Fine-tuned evaluation
5. Auto-push results to GitHub

**Requirements**: Kaggle GPU T4 x2, MVTec AD dataset attached.

## Cell 1 — Setup & Install

In [ ]:
!git clone https://github.com/dvydinh/vlm-industrial-finetuner.git
!pip install -q scikit-learn opencv-python-headless pandas numpy tqdm transformers peft bitsandbytes trl wandb

## Cell 2 — Data Preprocessing (ETL Pipeline)
Converts MVTec AD from unsupervised format to supervised instruction-tuning JSONL.
- Grayscale → RGB conversion
- 1024×1024 → 336×336 resize
- Stratified 80/20 train/test split
- Context-anchored prompts with item category tagging

In [ ]:
!python vlm-industrial-finetuner/src/data_builder.py \
    --data_dir /kaggle/input/datasets/ipythonx/mvtec-ad \
    --output_dir /kaggle/working/processed_data

## Cell 3 — Zero-Shot Baseline Evaluation
Run the base LLaVA-1.5-7B (no fine-tuning) to establish a performance floor.

Results auto-saved to `/kaggle/working/results/eval_baseline.json`

In [ ]:
!python vlm-industrial-finetuner/src/evaluate.py \
    --baseline \
    --test_data /kaggle/working/processed_data

## Cell 4 — QLoRA Fine-Tuning
Train LoRA adapters on Q-K-V-O self-attention projections.

Training logs auto-saved to `/kaggle/working/results/training_log.json`

In [ ]:
!python vlm-industrial-finetuner/src/train.py \
    --dataset /kaggle/working/processed_data \
    --output_dir /kaggle/working/lora_weights \
    --epochs 5 \
    --lr 2e-5

## Cell 5 — Fine-Tuned Evaluation
Merge LoRA adapter into base model and evaluate on the same test set.

Results auto-saved to `/kaggle/working/results/eval_finetuned.json`

In [ ]:
!python vlm-industrial-finetuner/src/evaluate.py \
    --model_dir /kaggle/working/lora_weights \
    --test_data /kaggle/working/processed_data

## Cell 6 — Auto-Push Results to GitHub
⚠️ **Replace `ghp_YOUR_TOKEN_HERE` with your GitHub Personal Access Token** (classic, `repo` scope).

Create token: GitHub → Settings → Developer settings → Personal access tokens → Classic

In [ ]:
import subprocess, os

# ═══ PASTE YOUR GITHUB TOKEN HERE ═══
GITHUB_TOKEN = "ghp_YOUR_TOKEN_HERE"

# Config git
subprocess.run(["git", "config", "--global", "user.email", "dvydinh@users.noreply.github.com"], check=True)
subprocess.run(["git", "config", "--global", "user.name", "dvydinh"], check=True)

# Copy results into repo
os.makedirs("vlm-industrial-finetuner/results", exist_ok=True)
subprocess.run("cp /kaggle/working/results/*.json vlm-industrial-finetuner/results/", shell=True)
subprocess.run("cp /kaggle/working/results/*.csv vlm-industrial-finetuner/results/", shell=True)

# Commit & Push
os.chdir("vlm-industrial-finetuner")
subprocess.run(["git", "add", "results/"], check=True)
subprocess.run(["git", "commit", "-m", "results: auto-upload evaluation metrics from Kaggle"], check=True)
subprocess.run(["git", "push", f"https://{GITHUB_TOKEN}@github.com/dvydinh/vlm-industrial-finetuner.git", "main"], check=True)
print("✅ Results pushed to GitHub! Run 'git pull' on local.")